# Heart Monitor recordings

Development-only exploration of `heart_capture_v1` recordings. Select **Heart Monitor (local)** as the VS Code kernel, then **Run All**. Raw files are never overwritten. All rates and peaks are candidates, not clinical measurements.

Nominal time is based on stored sample indices, not a hardware sequence. FIFO timestamps describe host batch reads, not individual sample timing; their differences are not sampling jitter. Interrupted recordings can contain unknown loss even when stored indices are contiguous.

In [2]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

REPO_ROOT = next((parent for parent in (Path.cwd(), *Path.cwd().parents)
                  if (parent / 'tools/heart-monitor/heart_analysis.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Run the notebook with its working directory inside the repository')
sys.path.insert(0, str(REPO_ROOT / 'tools/heart-monitor'))
from heart_analysis import load_capture, analyze_capture

pio.templates.default = 'plotly_white'
pio.renderers.default = 'plotly_mimetype'
print('Local interpreter:', sys.executable)

Local interpreter: /home/jan/dev/esp32-macropad/.venv-heart/bin/python


## Recording selection

Choose a directory and recording below. The default is the most recently modified `capture*.csv` in `build/temp`. Change `CAPTURE_PATH` to inspect another file, including an older extensionless download. Keep durable originals outside `build/`, which is ignored by Git and may be deleted during cleanup.

Record finger placement, motion, reference reading and its approximate time in `SESSION_NOTES`. An optional watch reference is displayed only; it does not affect processing.

In [3]:
RECORDING_DIR = REPO_ROOT / 'build/temp'
recordings = sorted(RECORDING_DIR.glob('capture*.csv'))
if not recordings:
    raise FileNotFoundError(f'No capture*.csv files in {RECORDING_DIR}; choose your recording directory')
display(pd.DataFrame([{'filename': path.name, 'bytes': path.stat().st_size,
                       'modified': pd.to_datetime(path.stat().st_mtime, unit='s')}
                      for path in recordings]))
CAPTURE_PATH = max(recordings, key=lambda path: path.stat().st_mtime)
SESSION_NOTES = ''
REFERENCE_BPM = None

capture = load_capture(CAPTURE_PATH)
print('Selected:', capture.path)
print('Session notes:', SESSION_NOTES or '(not recorded)')
display(pd.DataFrame(capture.metadata.items(), columns=['metadata', 'value']))
display(pd.DataFrame({'acquisition_warning': capture.warnings or ['No acquisition warnings found']}))
display(capture.samples[['red', 'ir']].agg(['min', 'max', 'mean', 'std']).round(2))

,filename,bytes,modified
0,capture-e30050af.csv,226663,2026-10-12 19:31:15.164537191


Selected: /home/jan/dev/esp32-macropad/build/temp/capture-e30050af.csv
Session notes: (not recorded)


,metadata,value
0,format,heart_capture_v1
1,channels,"red,ir"
2,sensor_address,0x57
3,part_id,0x15
4,revision,6
5,sensor_sample_rate_hz,100
6,sample_average,1
7,effective_sample_rate_hz,100
8,pulse_width_us,411
9,adc_range_na,4096


,acquisition_warning
0,No acquisition warnings found


,red,ir
min,33.00,57903.00
max,105209.00,115399.00
mean,103206.12,113685.45
std,1577.70,1001.46


In [3]:
raw = capture.samples
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=['Red', 'IR'])
for row, channel in enumerate(('red', 'ir'), start=1):
    fig.add_trace(go.Scattergl(x=raw.time_s, y=raw[channel], name=channel, mode='lines'), row=row, col=1)
    fig.update_yaxes(title_text='Raw counts', row=row, col=1)
fig.update_xaxes(title_text='Nominal time (s)', row=2, col=1)
fig.update_layout(height=600, title=f'Raw optical signals: {capture.path.name}')
fig.show()

poll = make_subplots(rows=2, cols=1, shared_xaxes=True)
poll.add_trace(go.Scattergl(x=raw.time_s, y=(raw.read_elapsed_us - raw.nominal_time_us) / 1000,
                           mode='lines', name='Host minus nominal'), row=1, col=1)
poll.add_trace(go.Scattergl(x=raw.time_s, y=raw.read_elapsed_us.diff() / 1000,
                           mode='lines', name='Poll timestamp increment'), row=2, col=1)
poll.update_yaxes(title_text='Host minus nominal (ms)', row=1, col=1)
poll.update_yaxes(title_text='Poll increment (ms)', row=2, col=1)
poll.update_xaxes(title_text='Nominal time (s)', row=2, col=1)
poll.update_layout(height=450, title='Host FIFO polling: batching and clock drift, not sample jitter')
poll.show()

## Exploratory processing

Adjust the parameters below, then rerun this and subsequent code cells. The third-order bandpass uses offline zero-phase filtering; it is not a causal device algorithm. Startup is excluded from processing. Shaded filter edges remain plotted but are excluded from summaries.

Peak distance, prominence and polarity are heuristic. They can miss beats or count extra peaks. Red/IR or spectral/peak disagreements must remain visible; no automatic contact classifier, SpO2, rhythm diagnosis or validated HRV is provided.

In [4]:
SETTINGS = dict(startup_s=1.0, low_hz=0.4, high_hz=4.0, edge_s=2.0,
                min_distance_s=0.35, prominence_sigma=1.0, window_s=10.0, invert=False)
analysis = analyze_capture(capture, **SETTINGS)
display(pd.DataFrame(SETTINGS.items(), columns=['parameter', 'value']))
display(analysis['summary'].round(3))

processed = analysis['signals']
beats = analysis['beats']
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=['Red', 'IR'])
for row, channel in enumerate(('red', 'ir'), start=1):
    fig.add_trace(go.Scattergl(x=processed.time_s, y=processed[f'{channel}_filtered'],
                              name=f'{channel} filtered', mode='lines'), row=row, col=1)
    for eligible, label, color in ((True, 'candidate peaks', 'black'), (False, 'excluded edge peaks', 'gray')):
        selected = beats.loc[(beats.channel == channel) & (beats.eligible == eligible)]
        fig.add_trace(go.Scattergl(x=selected.time_s, y=selected.value, mode='markers',
                                  name=f'{channel} {label}', marker=dict(color=color, size=6)), row=row, col=1)
    fig.update_yaxes(title_text='Filtered counts', row=row, col=1)
eligible_times = processed.loc[processed.eligible, 'time_s']
for start, end in ((processed.time_s.iloc[0], eligible_times.iloc[0]),
                   (eligible_times.iloc[-1], processed.time_s.iloc[-1])):
    fig.add_vrect(x0=start, x1=end, fillcolor='gray', opacity=0.15, line_width=0, layer='below')
fig.update_xaxes(title_text='Nominal time (s)', row=2, col=1)
fig.update_layout(height=600, title='Filtered signals and candidate peaks; gray edges excluded from summaries')
fig.show()

,parameter,value
0,startup_s,1.0
1,low_hz,0.4
2,high_hz,4.0
3,edge_s,2.0
4,min_distance_s,0.35
5,prominence_sigma,1.0
6,window_s,10.0
7,invert,False


,channel,baseline_counts,filtered_rms_counts,filtered_rms_percent,spectral_candidate_bpm,peak_interval_candidate_bpm,eligible_peaks
0,red,103171.756,33.689,0.033,47.393,71.429,74
1,ir,113659.094,75.028,0.066,47.393,48.780,51


In [5]:
spectrum = analysis['spectrum']
fig = go.Figure()
for channel in ('red', 'ir'):
    selected = spectrum.loc[spectrum.channel == channel]
    maximum = selected.power.max()
    relative_power = selected.power / maximum if maximum > 0 else selected.power * 0
    fig.add_trace(go.Scatter(x=selected.frequency_hz, y=relative_power, name=channel, mode='lines'))
fig.update_layout(height=450, title='Hann-windowed spectrum; each channel normalized separately',
                  xaxis_title='Frequency (Hz)', yaxis_title='Relative power')
fig.show()

In [6]:
windows = analysis['windows']
display(windows.round(3))
if windows.empty:
    print('No full windows: reduce window_s or record longer. Raw signals and the whole-record spectrum are still available.')
else:
    rates = windows.melt(id_vars=['channel', 'start_s'],
                         value_vars=['spectral_candidate_bpm', 'peak_interval_candidate_bpm'],
                         var_name='method', value_name='candidate_bpm')
    fig = px.line(rates, x='start_s', y='candidate_bpm', color='method', facet_row='channel',
                  markers=True, height=500, title='Windowed candidate rates: disagreement is diagnostic',
                  labels={'start_s': 'Window start (nominal s)', 'candidate_bpm': 'Candidate BPM'})
    if REFERENCE_BPM is not None:
        fig.add_hline(y=REFERENCE_BPM, line_dash='dash', annotation_text='Approximate watch reference')
    fig.show()

intervals = analysis['intervals']
if intervals.empty:
    print('No candidate intervals with these parameters')
else:
    fig = px.scatter(intervals, x='time_s', y='interval_s', color='channel', symbol='eligible',
                     height=450, title='Candidate intervals: not ECG or validated HRV',
                     labels={'time_s': 'Nominal time (s)', 'interval_s': 'Candidate interval (s)'})
    fig.show()

,channel,start_s,end_s,spectral_candidate_bpm,peak_interval_candidate_bpm,intervals,red_ir_correlation
0,red,3.0,13.0,138.0,105.263,15,0.739
1,red,13.0,23.0,48.0,68.966,11,0.710
2,red,23.0,33.0,48.0,85.714,12,0.760
3,red,33.0,43.0,48.0,64.865,10,0.853
4,red,43.0,53.0,48.0,68.966,13,0.741
5,ir,3.0,13.0,96.0,81.081,11,0.739
6,ir,13.0,23.0,48.0,46.154,7,0.710
7,ir,23.0,33.0,48.0,46.154,7,0.760
8,ir,33.0,43.0,48.0,47.809,6,0.853
9,ir,43.0,53.0,48.0,48.193,8,0.741


In [7]:
intervals = analysis['intervals']
if intervals.empty:
    print('No detected beat intervals to plot with these parameters')
else:
    fig = go.Figure()
    for channel, color in (('red', '#c43c39'), ('ir', '#16837a')):
        channel_intervals = intervals.loc[intervals.channel == channel]
        detected = channel_intervals.loc[channel_intervals.eligible]
        excluded = channel_intervals.loc[~channel_intervals.eligible]
        fig.add_trace(go.Scatter(
            x=detected.time_s, y=detected.candidate_bpm, mode='lines+markers',
            name=f'{channel}: per detected interval',
            line=dict(color=color, width=1), marker=dict(size=5), opacity=0.65,
            customdata=detected[['interval_s']],
            hovertemplate='Time %{x:.2f} s<br>BPM %{y:.1f}<br>Interval %{customdata[0]:.3f} s<extra>%{fullData.name}</extra>',
        ))
        fig.add_trace(go.Scatter(
            x=detected.time_s,
            y=detected.candidate_bpm.rolling(5, min_periods=3).median(),
            mode='lines', name=f'{channel}: rolling median (5 intervals)',
            line=dict(color=color, width=3, dash='dash'),
        ))
        fig.add_trace(go.Scatter(
            x=excluded.time_s, y=excluded.candidate_bpm, mode='markers',
            name=f'{channel}: excluded filter edges',
            marker=dict(color='gray', symbol='x', size=7),
        ))
    if REFERENCE_BPM is not None:
        fig.add_hline(y=REFERENCE_BPM, line_dash='dot', annotation_text='Approximate watch reference')
    fig.update_layout(
        height=500, title='Detected-peak BPM: 60 / interval; candidate measurements',
        xaxis_title='Nominal time of second detected peak (s)',
        yaxis_title='Detected-interval BPM', hovermode='closest',
    )
    fig.show()

## Interpretation

Compare raw and filtered traces, both channels, and both rate methods. A dominant harmonic or extra peaks can produce doubled rates. Small modulation can still contain useful periodicity; a spectrum alone does not validate each detected beat. Ten-second spectral windows have coarse frequency resolution, about 6 BPM between bins. Shortening windows further reduces precision.

Treat a watch reading as an approximate, nonsynchronized reference, not a target to tune thresholds against. Save placement, motion and reference-time notes alongside your recordings. These outputs do not establish clinical accuracy, SpO2, AFib or validated heart-rate variability.

In [4]:
from replay import replay_capture
import importlib
import replay as replay_module
importlib.reload(replay_module)

REPLAY_SPEED = 0  # 0 = accelerated; 1 = real-time pacing
live = replay_module.replay_capture(capture, speed=REPLAY_SPEED)
display(live.iloc[::max(1, round(capture.sample_rate_hz))][
    ['time_s', 'state', 'candidate_bpm', 'bpm', 'correlation', 'rms_counts']])

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=['C++ streaming BPM (gaps = unavailable)',
                                    'Periodicity strength (heuristic)',
                                    'Causal IR waveform and accepted peaks',
                                    'Pulse binding: 200 ms ON'])
fig.add_trace(go.Scatter(x=live.time_s, y=live.bpm, name='Live BPM',
                         connectgaps=False, line={'color': '#16837a'}), row=1, col=1)
fig.add_trace(go.Scatter(x=live.time_s, y=live.correlation, name='Correlation'), row=2, col=1)
fig.add_trace(go.Scatter(x=live.time_s, y=live.waveform, name='Filtered IR'), row=3, col=1)
detected = live.loc[(live.beat_count.diff() > 0) & (live.pulse == 1)]
fig.add_trace(go.Scatter(x=detected.time_s, y=detected.waveform, mode='markers',
                         name='Accepted candidate peaks', marker={'color': '#c43c39'}), row=3, col=1)
fig.add_trace(go.Scatter(x=live.time_s, y=live.pulse, name='Pulse',
                         line={'shape': 'hv', 'color': '#c43c39'}), row=4, col=1)
if REFERENCE_BPM is not None:
    fig.add_hline(y=REFERENCE_BPM, line_dash='dot', row=1, col=1)
fig.update_yaxes(title_text='BPM', row=1, col=1)
fig.update_yaxes(title_text='Correlation', range=[0, 1], row=2, col=1)
fig.update_yaxes(title_text='Counts', row=3, col=1)
fig.update_yaxes(title_text='ON/OFF', range=[-0.1, 1.1], row=4, col=1)
fig.update_xaxes(title_text='Nominal time (s)', row=4, col=1)
fig.update_layout(height=850, title='Firmware-equivalent C++ replay: past samples only')
fig.show()
print('Quality and detections are experimental; the sparkline retains only coarse time buckets.')

,time_s,state,candidate_bpm,bpm,correlation,rms_counts
0,0.0,measuring,NaN,NaN,0.000000,0.000000
100,1.0,measuring,NaN,NaN,0.000000,0.000000
200,2.0,measuring,NaN,NaN,0.000000,0.000000
300,3.0,measuring,NaN,NaN,0.000000,0.000000
400,4.0,measuring,NaN,NaN,0.000000,0.000000
...,...,...,...,...,...,...
5700,57.0,usable,46.153847,46.153847,0.911448,62.423714
5800,58.0,usable,46.511627,46.511627,0.839818,59.369816
5900,59.0,usable,46.153847,46.153847,0.581967,67.305367
6000,60.0,weak_signal,47.619049,NaN,0.541042,70.475174


Quality and detections are experimental; the sparkline retains only coarse time buckets.
